<a href="https://colab.research.google.com/github/Joy-Kimani/slm-arm64-benchmark/blob/main/03_export_to_gguf(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Export the trained 70M model for llama.cpp

This notebook takes our trained 70M param Llama model checkpoint it wraps the raw weights into a standard Hugging Face LlamaForCausalLM model. After confirming that numerical parity and validation loss remain intact, it exports quantized GGUF files (F16, Q8_0, and Q4_0) for fast local CPU inference with llama.cpp.

Prerequisite: Run 02_train_model first so that best_model_params.pt, tokenizer.json, meta.json, and validation.bin are available in Drive.

## Step 1: Mount Google Drive and set the paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import numpy as np
import torch
import torch.nn.functional as F

out_dir = '/content/drive/MyDrive/slm70m'
export_dir = os.path.join(out_dir, 'hf_export')

meta = json.load(open(os.path.join(out_dir, 'meta.json')))
device = "cuda" if torch.cuda.is_available() else "cpu"

Mounted at /content/drive


In [ ]:
!pip install -q transformers safetensors

## Step 2: Load the trained weights into a Hugging Face Llama model

The layer names in our model already follow the Hugging Face Llama naming, so the weights load without renaming anything.

In [ ]:
from tokenizers import Tokenizer
from transformers import LlamaConfig, LlamaForCausalLM

tokenizer = Tokenizer.from_file(os.path.join(out_dir, 'tokenizer.json'))
eot_id = tokenizer.token_to_id("<|endoftext|>")

config = LlamaConfig(
    vocab_size=meta['vocab_size'],
    hidden_size=640,
    intermediate_size=1728,
    num_hidden_layers=12,
    num_attention_heads=10,
    num_key_value_heads=10,
    max_position_embeddings=256,
    rms_norm_eps=1e-5,
    rope_theta=10000.0,
    hidden_act='silu',
    attention_bias=False,
    mlp_bias=False,
    tie_word_embeddings=True,
    bos_token_id=None,
    eos_token_id=eot_id,
)

state_dict = torch.load(os.path.join(out_dir, 'best_model_params.pt'), map_location='cpu')

hf_model = LlamaForCausalLM(config)
hf_model.load_state_dict(state_dict)
hf_model = hf_model.to(device)
hf_model.eval()

print(sum(p.numel() for p in hf_model.parameters()) / 1e6, "M parameters")

69.72992 M parameters


## Step 3: Check the exported model perfoms the same

Next, we'll test the model to make sure its error score (validation loss) is still around 1.39, just like when it finished training. If that number jumps much higher, it's a sign that something broke during the transfer.

In [ ]:
data = np.memmap(os.path.join(out_dir, 'validation.bin'), dtype=np.uint16, mode='r')

torch.manual_seed(0)
losses = []
with torch.no_grad():
    for _ in range(20):
        ix = torch.randint(len(data) - 256, (16,))
        x = torch.stack([torch.from_numpy((data[i:i+256]).astype(np.int64)) for i in ix]).to(device)
        y = torch.stack([torch.from_numpy((data[i+1:i+257]).astype(np.int64)) for i in ix]).to(device)
        logits = hf_model(x).logits
        losses.append(F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1)).item())

print("validation loss:", np.mean(losses))

validation loss: 1.4095261991024017


## Step 4: Save the model and the tokenizer

In [ ]:
from transformers import PreTrainedTokenizerFast

os.makedirs(export_dir, exist_ok=True)
hf_model.save_pretrained(export_dir)

hf_tokenizer = PreTrainedTokenizerFast(tokenizer_file=os.path.join(out_dir, 'tokenizer.json'), eos_token='<|endoftext|>')
hf_tokenizer.save_pretrained(export_dir)

print(os.listdir(export_dir))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json']


## Step 5: Generate a story from the exported model

This is the text we will compare against when the GGUF(compressed) versions run later.


In [ ]:
prompt = "Once upon a time there was a pumpkin."
input_ids = torch.tensor([hf_tokenizer.encode(prompt)]).to(device)
out = hf_model.generate(input_ids, max_new_tokens=100, do_sample=False, pad_token_id=eot_id)
print(hf_tokenizer.decode(out[0]))

Once upon a time there was a pumpkin. It was a very special pumpkin because it was so big and round. It was so big that it could fit in the garden.

One day, a little girl saw the pumpkin. She was so excited and she wanted to take it home. She asked her mom if she could take it home.

Her mom said yes and the little girl was so happy. She took the pumpkin home and she put it in her room.

Every day, the little girl would look at the


## Step 6: Convert to GGUF

llama.cpp uses its own file format called GGUF. The converter script comes with llama.cpp.

To run our model efficiently on a regular computer, llama.cpp needs it packaged into a special file format called GGUF. We will use the built-in conversion tool included with llama.cpp to create this file. Since we are locking in the exact version of llama.cpp we tested for stability, make sure to jot that version number down for your results file later.

In [ ]:
!git clone https://github.com/ggml-org/llama.cpp
!git -C llama.cpp checkout f8caa5ea30b5775512906ae3b7ee53b21a7a6e1d
!pip install -q gguf sentencepiece protobuf

Cloning into 'llama.cpp'...
remote: Enumerating objects: 167440, done.
remote: Counting objects: 100% (168/168), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 167440 (delta 94), reused 78 (delta 76), pack-reused 167272 (from 3)
Receiving objects: 100% (167440/167440), 495.32 MiB | 26.30 MiB/s, done.
Resolving deltas: 100% (120495/120495), done.
Updating files: 100% (4609/4609), done.
Note: switching to 'f8caa5ea30b5775512906ae3b7ee53b21a7a6e1d'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is no

The conversion tool identifies how a model splits text (its tokenizer) by checking against a built-in list of known vocabularies. Because we trained our own custom vocabulary, the tool won't recognize it and will stop with an error: "BPE pre-tokenizer was not recognized". Since our custom tokenizer still follows the standard GPT-2 rules for breaking up words, we simply pass a setting that tells the converter to treat it like "gpt-2".

In [ ]:
marker = 'raise NotImplementedError("BPE pre-tokenizer was not recognized'

for conv in ['llama.cpp/conversion/base.py', 'llama.cpp/convert_hf_to_gguf.py']:
    if os.path.exists(conv) and marker in open(conv).read():
        src = open(conv).read()
        old = '        if res is None:\n            logger.warning("\\n")'
        new = '        if res is None:\n            res = "gpt-2"\n        if res is None:\n            logger.warning("\\n")'
        assert old in src
        open(conv, 'w').write(src.replace(old, new, 1))
        print("patched", conv)
        break

patched llama.cpp/conversion/base.py


In [ ]:
!python llama.cpp/convert_hf_to_gguf.py {export_dir} --outfile {out_dir}/slm70m-f16.gguf --outtype f16

## Step 7: Build the quantize tool and make the smaller versions

To make our model smaller and faster, we use compression methods called Q8_0 (which shrinks each piece of model data down to 8 bits) and Q4_0 (which shrinks it to 4 bits). Both work smoothly with our model because our layer sizes (640 and 1728) divide evenly by 32. Other popular compression formats, like Q4_K_M, require sizes that divide evenly by 256. Since ours don't fit that math, the tool would end up skipping that compression on most layers anyway—which is why we stick to Q8_0 and Q4_0.

We limit the build to 2 parallel jobs. Without a limit the compiler can use up all the RAM on Colab and crash the session.

In [ ]:
!cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release
!cmake --build llama.cpp/build -j 2 --target llama-quantize

-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 1.0.0-dev
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.43.0")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found Ope

In [ ]:
!llama.cpp/build/bin/llama-quantize {out_dir}/slm70m-f16.gguf {out_dir}/slm70m-q8_0.gguf Q8_0
!llama.cpp/build/bin/llama-quantize {out_dir}/slm70m-f16.gguf {out_dir}/slm70m-q4_0.gguf Q4_0

version: 1.0.0-dev (build 12548, commit f8caa5ea3)
built with GNU 13.3.0 for Linux x86_64
llama_quantize: quantizing '/content/drive/MyDrive/slm70m/slm70m-f16.gguf' to '/content/drive/MyDrive/slm70m/slm70m-q8_0.gguf' as Q8_0
llama_model_loader: loaded meta data with 24 key-value pairs and 110 tensors from /content/drive/MyDrive/slm70m/slm70m-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Hf_Export
llama_model_loader: - kv   3:                         general.size_label str              = 70M
llama_model_loader: - kv   4:                          llama.block_count u32              = 12
llama_model_loader: - kv   5:    

In [ ]:
!ls -lh {out_dir}/*.gguf

-rw------- 1 root root 134M Nov 19 12:29 /content/drive/MyDrive/slm70m/slm70m-f16.gguf
-rw------- 1 root root  43M Nov 21 09:48 /content/drive/MyDrive/slm70m/slm70m-q4_0.gguf
-rw------- 1 root root  72M Nov 21 09:48 /content/drive/MyDrive/slm70m/slm70m-q8_0.gguf
